# Transformers (BERT)

## Timeline:

*   2017: the Transformer paper introduces the encoder–decoder architecture. [Paper Link](https://proceedings.neurips.cc/paper_files/paper/2017/file/3f5ee243547dee91fbd053c1c4a845aa-Paper.pdf).
*   June 2018: OpenAI's GPT: take only the decoder half, stack it, optimize for generating text.
*   October 2018: Google's BERT: take only the encoder half, stack it, optimize for understanding text.

![My Image](https://miro.medium.com/v2/resize:fit:4800/format:webp/1*Qww2aaIdqrWVeNmo3AS0ZQ.png)

---
## Encoder-Only vs Decoder-Only Models

| | **Encoder-Only (BERT)** | **Decoder-Only (GPT)** |
|---|---|---|
| **Goal** | Make predictions about words *within* an input sequence | Predict a new output sequence in response to an input |
| **Built by** | Dropping the decoder, stacking Transformer encoders | Dropping the encoder, stacking Transformer decoders |
| **What it produces** | Rich numeric vector representations for each token | New tokens, one at a time |
| **Training task** | Masked Language Modelling: fill in the blank | Next Token Prediction: what comes next |
| **Context direction** | Bidirectional (left + right) | Unidirectional (left only) |
| **Takes a prompt?** | No: takes a sequence to make a prediction about | Yes: the prompt is the input |
| **Best at** | NLU: sentiment analysis, NER, classification, search | NLG: chatbots, translation, code generation, summarisation |
| **Familiar examples** | BERT, RoBERTa, DistilBERT, ALBERT | GPT-3.5, GPT-4, Claude, Llama |
| **How you adapt it** | Fine-tune on labelled data | Often just write a better prompt |

> **In one line:** encoders are built to *understand* text; decoders are built to *continue* it.

---
## Pre-training and Fine-tuning

Before 2018, each new NLP task meant designing and training a new model from scratch. GPT popularised a different approach, and BERT adopted it wholesale. It is now the dominant paradigm for every large language model.

The idea splits training into **two stages**:

**Stage 1: Pre-training.** Train one large model on enormous amounts of raw text to acquire a broad understanding of language: word usage, grammar, facts, and how sentences relate. The result is a task-agnostic **foundational model**. It knows a lot about language but is not yet useful for any particular job. This is slow and expensive: BERT Base took around 4 days on 16 TPUs, BERT Large around 4 days on 64 TPUs.

**Stage 2: Fine-tuning.** Take a copy of that foundational model, attach a small output layer called a **classification head**, and train it further on a small labelled dataset for your specific task. This takes minutes to hours, not days.

---
## How BERT Was Pre-trained

BERT was trained on raw, unlabelled text: English Wikipedia (2,500M words) plus 11,038 books from BookCorpus (800M words). No human annotation. Two objectives were designed to force the model to use context from **both** directions.

### 1. Masked Language Modelling (MLM)

Randomly hide 15% of the tokens and train BERT to predict what's missing.

> `A man was fishing on the river` → input `A man was [MASK] on the river`, target `fishing`

BERT predicts only the missing word, not the whole sentence. Loss is computed at the masked positions only.

**The 80/10/10 detail.** Of the 15% chosen, the token is replaced with `[MASK]` 80% of the time, a *random* token 10% of the time, and left *unchanged* 10% of the time. The last 10% is the clever bit: because the model can never be sure which position is being tested, it must build a good representation for **every** token, not just the visibly masked ones.

### 2. Next Sentence Prediction (NSP)

Given two segments, classify whether the second logically follows the first: `IsNext` or `NotNext`.

> Format: `[CLS] segment A [SEP] segment B [SEP]`

Training data is free: take the real next sentence 50% of the time, a random one the other 50%. The motivation was that tasks like question answering and natural language inference depend on the relationship *between* sentences, which word-level modelling doesn't capture.


### 3. Why this makes BERT bidirectional

Both tasks only make sense if the model can see the whole sequence at once. Predicting a masked word in the middle means reading the words to its left **and** its right.

Try it yourself:

| What you see | Your guess |
|---|---|
| `A man was ____` | could be anything |
| `____ on the river` | could be anything |
| `A man was ____ on the river` | **fishing** |


### 4. The payoff: contextual embeddings

Older methods like Word2Vec gave each word **one fixed vector**. "Bank" in *river bank* and *bank account* would be identical.

Because BERT reads the full sentence, it produces a **different vector for the same word depending on its neighbours**:

> `I deposited money at the **bank**` → finance vector
> `We sat on the river **bank**` → geography vector

Same spelling, different meaning, different numbers. This is what "deep language understanding" actually means in practice, and it is why a BERT vector is a far better input to a classifier than a Word2Vec one.

---

## Text Classification Using BERT:

![My Image](https://www.researchgate.net/profile/Juan-Pablo-Usuga-Cadavid/publication/353419108/figure/fig1/AS:1052388236476416@1627920316521/Example-of-a-trained-BERT-for-text-classification_W640.jpg)

In [21]:
# Libraries

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
import torch
from sklearn.metrics import accuracy_score
import numpy as np

In [ ]:
# Load BERT-Base Model

bert_model = AutoModel.from_pretrained("bert-base-uncased")

In [ ]:
#Load Dataset

dataset = load_dataset("fancyzhx/ag_news")

# Labels:
# 0 → World
# 1 → Sports
# 2 → Business
# 3 → Sci/Tech

train_dataset = dataset["train"].shuffle(seed=42).select(range(2000))
eval_dataset = dataset["test"].shuffle(seed=42).select(range(500))

In [ ]:
# Load BERT's tokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

### Contextual Word Embeddings from BERT

In [6]:
sample = train_dataset[0]["text"]
print(sample)

Bangladesh paralysed by strikes Opposition activists have brought many towns and cities in Bangladesh to a halt, the day after 18 people died in explosions at a political rally.


In [7]:
inputs = tokenizer(
    sample,
    return_tensors="pt",
    truncation=True,
    max_length=128
)

# Returns:
# input_ids → Token to id from BERT's predefined vocabulary
# token_type_ids → Identify which sentence a token belongs to
# attention_mask → Identify which tokens to pay attention: 0 for PAD, else 1

inputs

{'input_ids': tensor([[  101,  7269, 11498,  2135,  6924,  2011,  9326,  4559, 10134,  2031,
          2716,  2116,  4865,  1998,  3655,  1999,  7269,  2000,  1037,  9190,
          1010,  1996,  2154,  2044,  2324,  2111,  2351,  1999, 18217,  2012,
          1037,  2576,  8320,  1012,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [8]:
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

print(tokens)

['[CLS]', 'bangladesh', 'para', '##ly', '##sed', 'by', 'strikes', 'opposition', 'activists', 'have', 'brought', 'many', 'towns', 'and', 'cities', 'in', 'bangladesh', 'to', 'a', 'halt', ',', 'the', 'day', 'after', '18', 'people', 'died', 'in', 'explosions', 'at', 'a', 'political', 'rally', '.', '[SEP]']


In [9]:
# Get BERT representations

with torch.no_grad():
    outputs = bert_model(**inputs)


In [10]:
# Get contextual embeddings

embeddings = outputs.last_hidden_state[0]

embeddings

tensor([[-0.2284,  0.0113, -0.0286,  ..., -0.5718,  0.2943, -0.1053],
        [ 0.6273, -0.1440, -0.4108,  ..., -0.1198,  0.2130,  0.1325],
        [ 0.1795, -0.4641, -0.0215,  ..., -1.0563, -0.3542, -0.2049],
        ...,
        [ 0.6749, -0.8151,  0.7201,  ..., -0.2991, -0.6498, -0.3273],
        [ 0.5202,  0.0062, -0.4603,  ..., -0.3126, -0.4497, -0.0898],
        [ 0.4117,  0.0501, -0.4302,  ..., -0.3381, -0.5367, -0.1679]])

### News Classification using BERT

In [11]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=128
    )

In [ ]:
train_dataset = train_dataset.map(tokenize, batched=True)
eval_dataset = eval_dataset.map(tokenize, batched=True)

In [14]:
train_dataset[0]

{'text': 'Bangladesh paralysed by strikes Opposition activists have brought many towns and cities in Bangladesh to a halt, the day after 18 people died in explosions at a political rally.',
 'label': 0,
 'input_ids': [101,
  7269,
  11498,
  2135,
  6924,
  2011,
  9326,
  4559,
  10134,
  2031,
  2716,
  2116,
  4865,
  1998,
  3655,
  1999,
  7269,
  2000,
  1037,
  9190,
  1010,
  1996,
  2154,
  2044,
  2324,
  2111,
  2351,
  1999,
  18217,
  2012,
  1037,
  2576,
  8320,
  1012,
  102],
 'token_type_ids': [0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0],
 'attention_mask': [1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1]}

In [ ]:
# Rename label → labels

train_dataset = train_dataset.rename_column("label", "labels")
eval_dataset = eval_dataset.rename_column("label", "labels")

In [15]:
# Dynamic padding

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

In [ ]:
# Load BERT for classification

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=4
)

In [22]:
# Setting up accuracy for metric

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    predictions = np.argmax(predictions, axis=1)

    return {
        "accuracy": accuracy_score(labels, predictions)
    }

In [23]:
# Training configuration

training_args = TrainingArguments(
    output_dir="./bert_agnews",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    logging_steps=50,
    eval_strategy="epoch",
)

In [24]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [25]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.394608,0.388474,0.874000
2,0.231259,0.410104,0.892000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=250, training_loss=0.38372622680664065, metrics={'train_runtime': 86.3568, 'train_samples_per_second': 46.319, 'train_steps_per_second': 2.895, 'total_flos': 188094893296128.0, 'train_loss': 0.38372622680664065, 'epoch': 2.0})

In [26]:
# Prediction

label_names = dataset["train"].features["label"].names  # Creates human-readable class names

def predict(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}  # Move inputs to the same device (GPU/CPU) as the model
    pred_id = model(**inputs).logits.argmax(dim=1).item()  # Run BERT, get the four class scores, find the class with the highest score, and return its ID.
    return label_names[pred_id]

print(predict("The stock market rallied today after strong earnings reports."))

Business


## Comparison of BERT Variants


| | **BERT-base** | **BERT-large** | **DistilBERT** | **RoBERTa** | **ALBERT** | **ELECTRA-base** | **DeBERTa-v3** |
|---|---|---|---|---|---|---|---|
| **Layers** | 12 | 24 | 6 | 12 | 12 | 12 | 12 |
| **Attention Heads** | 12 | 16 | 12 | 12 | 12 | 12 | 12 |
| **Hidden Size** | 768 | 1024 | 768 | 768 | 768 | 768 | 768 |
| **Params** | 110M | 340M | 66M | 125M | 12M (shared) | 110M | 184M |
| **Training Task** | MLM + NSP | MLM + NSP | Distillation (mimics BERT) | MLM only, no NSP | MLM + Sentence Order Prediction | Replaced Token Detection | MLM + Disentangled Attention |
| **Key Idea** | Original bidirectional encoder | Deeper/wider BERT | Smaller, faster copy of BERT | Better-tuned BERT (more data, longer training) | Parameter sharing across layers | Discriminator spots swapped tokens instead of predicting masks | Separates content/position in attention |
| **Trade-off** | Baseline | More accurate, much heavier | Loses some accuracy for speed/size | Needs more compute/data to train | Slower per-param despite fewer weights | Efficient pretraining, competitive accuracy | Strong accuracy, extra complexity |
| **Best at** | General-purpose NLU baseline | Tasks needing max accuracy | Mobile/edge, low-latency inference | SOTA-ish NLU with same architecture as BERT | Memory-constrained deployment | Efficient pretraining on a budget | GLUE/SQuAD leaderboard-level tasks |
| **HF Model Name** | `bert-base-uncased` | `bert-large-uncased` | `distilbert-base-uncased` | `roberta-base` | `albert-base-v2` | `google/electra-base-discriminator` | `microsoft/deberta-v3-base` |